# SMART CITY TRAFFIC & SAFETY ANALYTICS – Notebook 



In [38]:
import pandas as pd

traffic_logs = [
    "id:501,zone:A1,vehicle:Car,speed:62,time:08:30,violations:[None],status:Smooth",
    "id:502,zone:A1,vehicle:Bike,speed:85,time:09:10,violations:[Helmet],status:Busy",
    "id:503,zone:B2,vehicle:Bus,speed:45,time:17:25,violations:[None],status:Smooth",
    "id:504,zone:C3,vehicle:Car,speed:110,time:14:15,violations:[Overspeed],status:Congested",
    "id:505,zone:A1,vehicle:Truck,speed:40,time:18:50,violations:[None],status:Smooth"
]

def parse_entry(entry):
    pieces = entry.split(",")
    bag = {}
    for f in pieces:
        k, v = f.split(":", 1)
        bag[k] = v
    bag["id"] = int(bag["id"])
    bag["speed"] = int(bag["speed"])
    viol = bag.get("violations", "").strip("[]")
    bag["violation"] = None if viol == "None" or viol == "" else viol
    if "violations" in bag:
        del bag["violations"]
    return bag

parsed = [parse_entry(x) for x in traffic_logs]
df = pd.DataFrame(parsed)
df

,id,zone,vehicle,speed,time,status,violation
0,501,A1,Car,62,08:30,Smooth,None
1,502,A1,Bike,85,09:10,Busy,Helmet
2,503,B2,Bus,45,17:25,Smooth,None
3,504,C3,Car,110,14:15,Congested,Overspeed
4,505,A1,Truck,40,18:50,Smooth,None


## Q2 Average speed per zone


In [39]:
avg_speed = df.groupby("zone")["speed"].mean()
avg_speed

zone
A1     62.333333
B2     45.000000
C3    110.000000
Name: speed, dtype: float64

## Q3 Peak hour (most entries)


In [40]:
df["hour"] = df["time"].str[:2]
peak_hour = df["hour"].value_counts().idxmax()
peak_hour

'08'

## Q4 Vehicles speed > 80


In [41]:
df[df["speed"] > 80]

,id,zone,vehicle,speed,time,status,violation,hour
1,502,A1,Bike,85,09:10,Busy,Helmet,09
3,504,C3,Car,110,14:15,Congested,Overspeed,14


## Q5 Count each violation type


In [42]:
viol_counts = df["violation"].dropna().value_counts()
viol_counts

violation
Helmet       1
Overspeed    1
Name: count, dtype: int64

## Q6 Safety index per zone (simple formula)


In [43]:
viols_per_zone = df.groupby("zone")["violation"].apply(lambda x: x.notna().sum())
avg_speed_zone = df.groupby("zone")["speed"].mean()
safety_index = avg_speed_zone / (1 + viols_per_zone)
safety_index

zone
A1    31.166667
B2    45.000000
C3    55.000000
dtype: float64

## Q7 Summary each vehicle category


In [44]:
vehicle_summary = df.groupby("vehicle").agg(
    count=("id", "count"),
    avg_speed=("speed", "mean"),
    max_speed=("speed", "max"),
    min_speed=("speed", "min"),
)
vehicle_summary

,count,avg_speed,max_speed,min_speed
vehicle,,,,
Bike,1,85.0,85,85
Bus,1,45.0,45,45
Car,2,86.0,110,62
Truck,1,40.0,40,40


## Q8 High congestion zones


In [45]:
high_cong = df[df["status"] == "Congested"]["zone"].unique()
high_cong

array(['C3'], dtype=object)

## Q9 Classify each log into time window


In [46]:
def classify_window(t):
    h = int(t[:2])
    if 5 <= h < 12:
        return "Morning"
    elif 12 <= h < 17:
        return "Afternoon"
    elif 17 <= h < 21:
        return "Evening"
    else:
        return "Night"
df["time_window"] = df["time"].apply(classify_window)
df[["time", "time_window"]]

,time,time_window
0,08:30,Morning
1,09:10,Morning
2,17:25,Evening
3,14:15,Afternoon
4,18:50,Evening


## Q10 Final zone-level report


In [ ]:
zones = sorted(df['zone'].unique())
rows = []
for z in zones:
    zdf = df[df['zone'] == z]
    total_veh = len(zdf)
    avg_speed = zdf['speed'].mean()
    violations = zdf['violation'].notna().sum()
    mode_vals = zdf['vehicle'].mode()
    common_vehicle = mode_vals.iat[0]
    if not mode_vals.empty else None
    safety_index = avg_speed / (1 + violations)
    if safety_index < 40:
         safety_cat = 'High Risk'
    elif safety_index < 70: 
        safety_cat = 'Moderate'
    else: 
        safety_cat = 'Safe'
    rows.append({'zone': z, 'total_vehicles': total_veh, 'avg_speed': avg_speed, 'violations': violations, 'common_vehicle': common_vehicle, 'safety_index': safety_index, 'safety_category': safety_cat})
final_report_basic = pd.DataFrame(rows).set_index('zone')
final_report_basic

,total_vehicles,avg_speed,violations,common_vehicle,safety_index,safety_category
zone,,,,,,
A1,3,62.333333,1,Bike,31.166667,High Risk
B2,1,45.000000,0,Bus,45.000000,Moderate
C3,1,110.000000,1,Car,55.000000,Moderate
